# 04 — Dataclasses, Properties, and Model Invariants

Dataclasses reduce boilerplate for structured records, while properties and post-initialization checks help preserve invariants. Understand the generated methods and how mutability affects design.

*Intermediate · 75–120 min*

## You will learn

- Create mutable and frozen dataclasses
- Use field(default_factory=...) for mutable values
- Validate dataclass fields with __post_init__
- Provide controlled computed attributes with @property

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Dataclass boilerplate reduction

A dataclass generates common methods such as __init__, __repr__, and value-based __eq__. It is useful for records holding data with modest behavior. Type annotations on fields are not automatic runtime validators.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    sku: str
    price: float
    quantity: int = 0

product = Product("P-10", 19.5, 3)
print(product)
print(product == Product("P-10", 19.5, 3))

**Reasoning prompt:** What makes a dataclass more readable than a handwritten class with many simple fields?

### 2. Mutable defaults and field factories

Every instance should own its own list or dictionary when that collection represents instance-specific state. Dataclasses require field(default_factory=list) rather than a shared mutable default. This prevents surprising coupling between records.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Playlist:
    title: str
    songs: list[str] = field(default_factory=list)

first = Playlist("Morning")
second = Playlist("Evening")
first.songs.append("Track A")
print(first.songs, second.songs)
assert not second.songs

**Reasoning prompt:** Why does default_factory call list separately for each new instance?

### 3. Validation and frozen records

__post_init__ runs after dataclass initialization and can enforce constraints. frozen=True prevents ordinary field assignment after construction, but does not deeply freeze mutable objects stored in fields. Use immutable types for genuinely immutable contents.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Discount:
    label: str
    rate: float
    def __post_init__(self):
        if not 0 <= self.rate <= 1:
            raise ValueError("rate must be in [0,1]")
    def apply(self, price):
        return price * (1 - self.rate)

coupon = Discount("spring", 0.15)
print(coupon.apply(100))

**Reasoning prompt:** Why might a frozen dataclass containing a list still permit changes to that list?

### 4. Properties and derived values

@property exposes a read-only computed attribute through natural attribute syntax. Use a property setter only when you need validation during assignment. Avoid expensive or surprising work inside properties.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        self._celsius = float(value)

t = Temperature(20)
print(t.celsius, t.fahrenheit)

**Reasoning prompt:** Why should fahrenheit be computed rather than stored redundantly?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/01_oop_and_collections_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/04', {1: 'a57ed9362e004ba0', 2: '2e45d989821fe886', 3: '90d5e890f6f4ad4b', 4: '53c1ad56e319a756'})

**Exercise 1.** A frozen dataclass can't be changed after creation.

In [ ]:
from dataclasses import dataclass, FrozenInstanceError

@dataclass(frozen=___)
class Point:
    x: int
    y: int

p = Point(1, 2)
try:
    p.x = 10
    outcome = "changed"
except FrozenInstanceError:
    outcome = "refused"
check(1, (outcome, p == Point(1, 2)))

**Exercise 2.** Each `Team` must get its own list: use `default_factory`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Team:
    name: str
    members: list = field(default_factory=___)

a, b = Team("a"), Team("b")
a.members.append("ana")
check(2, (a.members, b.members))

**Exercise 3.** Validate fields in `__post_init__`.

In [ ]:
from dataclasses import dataclass

@dataclass
class Item:
    name: str
    price: float

    def __post_init__(self):
        if ___:
            raise ValueError("price cannot be negative")

try:
    Item("bad", -1)
    result = "accepted"
except ValueError:
    result = "rejected"
check(3, (result, Item("pen", 1.5).price))

**Exercise 4.** A computed `@property`: `area` is derived from width and height.

In [ ]:
from dataclasses import dataclass

@dataclass
class Rectangle:
    width: float
    height: float

    @___
    def area(self):
        return self.width * self.height

r = Rectangle(3, 4)
r.width = 5
check(4, r.area)

## Common mistakes

- Dataclass type annotations do not validate types at runtime.
- frozen=True is not recursive deep immutability.
- Never use a shared mutable default to represent per-instance data.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why use field(default_factory=list)?

   - **a)** So each instance gets its own list
   - **b)** It's faster
   - **c)** Lists aren't allowed in dataclasses

**2.** Where do you validate dataclass fields?

   - **a)** __init__ (written by hand)
   - **b)** __repr__
   - **c)** __post_init__

**3.** What does frozen=True give you?

   - **a)** Immutable, hashable instances
   - **b)** Automatic validation
   - **c)** Faster attribute access

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/04', {1: '2a28d7884674', 2: 'f5bdfec142a3', 3: '7e19a27eef80'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [05 — Iterables, Iterators, and Protocols](05_iterables_and_iterators.ipynb)